<a href="https://colab.research.google.com/github/gerryfrank10/ComputerVision/blob/main/GoogleNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import torch.nn as nn
import numpy as np

In [7]:
'''
Google Net Inception Architecture (2014)
'''
class BaseConv2d(nn.Module):
  def __init__(self,in_channels, out_channels, **kwargs):
    super(self).__init__()
    self.conv = nn.Conv2d(in_channels, out_channels, **kwargs)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.conv(x)
    x = self.relu(x)
    return x

class InceptionModule(nn.Module):
  def __init__(self,in_channels, n1x1, n3x3, n3x3red,n5x5, n5x5red, pool_proj):
    super(self).__init__()
    self.b1 = nn.Sequential(
        nn.Conv2d(in_channels, n1x1, kernel_size=1),
        nn.ReLU(True)
    )
    self.b2 = nn.Sequential(
        BaseConv2d(in_channels, n3x3red, kernel_size=1,),
        BaseConv2d(n3x3red, n3x3, kernel_size=3, padding=1),
    )

    self.b3 = nn.Sequential(
        BaseConv2d(in_channels, n5x5red, kernel_size=1),
        BaseConv2d(n5x5red, n5x5, kernel_size=5, padding=2),
    )
    self.b4 = nn.Sequential(
        nn.MaxPool2d(3, stride=1, padding=1),
        BaseConv2d(in_channels, pool_proj, kernel_size=1),
    )

  def forward(self, x):
    x1 = self.b1(x)
    x2 = self.b2(x)
    x3 = self.b3(x)
    x4 = self.b4(x)
    return torch.cat([x1,x2,x3,x4], dim=1)

class AuxilaryClassifier(nn.Module):
  def __init__(self,in_channels, num_classes, dropout=0.7):
    super(self).__init__()
    self.pool = nn.AvgPool2d(kernel_size=5, stride=3)
    self.conv = nn.AvgPool2d(5, stride=3)
    self.relu = nn.ReLU(True)
    self.flatten = nn.Flatten()
    self.fc1 = nn.Linear(2048, 1024)
    self.dropout = nn.Dropout(dropout)
    self.fc2 = nn.Linear(1024, num_classes)

  def forward(self, x):
    x = self.pool(x)
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.fc2(x)
    return x

class GoogleNet(nn.Module):
  def __init__(self,use_aux=True):
    super(self).__init__()

    self.use_aux = use_aux
    # Block 1
    self.conv1 = BaseConv2d(3, 64, kernel_size=7, stride=2, padding=3)
    self.lnr1 = nn.LocalResponseNorm(5, alpha=0.0001, beta=0.75)
    self.maxpool1 = nn.MaxPool2d(3, stride=2)

    # Block 2
    self.conv2 = BaseConv2d(64, 64, kernel_size=1)
    self.conv3 = BaseConv2d(64, 192, kernel_size=3, padding=1)
    self.lnr2 = nn.LocalResponseNorm(5, alpha=0.0001, beta=0.75)
    self.maxpool2 = nn.MaxPool2d(3, stride=2, padding=1)

    # Block 3
    self.inception3a = InceptionModule(192, 64, 96, 128, 16, 32, 32)
    self.inception3b = InceptionModule(256, 128, 128, 192, 32, 96, 64)
    self.maxpool3 = nn.MaxPool2d(3, stride=2, padding=1)

    # Block 4
    self.inception4a = InceptionModule(480 , 192, 96, 208, 16, 48, 64)
    self.inception4b = InceptionModule(512, 160, 112, 224, 24, 64, 64)
    self.inception4c = InceptionModule(512, 128, 128, 256, 24, 64, 64)
    self.inception4d = InceptionModule(512, 112, 144, 288, 32, 64, 64)
    self.inception4e = InceptionModule(528, 256, 160, 320, 32, 128, 128)
    self.maxpool4 = nn.MaxPool2d(3, stride=2, padding=1)

    # block 5
    self.inception5a = InceptionModule(832, 256, 160, 320, 32, 128, 128)
    self.inception5b = InceptionModule(832, 384, 192, 384, 48, 128, 128)

    if self.use_aux:
      self.aux1 = AuxilaryClassifier(512, 1000)
      self.aux2 = AuxilaryClassifier(528, 1000)

    # Block 6
    self.avgpool = nn.AvgPool2d(7, stride=1)
    self.dropout = nn.Dropout(0.4)
    self.fc = nn.Linear(1024, 1000)

  def forward(self, x):
        ## block 1
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = self.lrn1(x)

        ## block 2
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.lrn2(x)
        x = self.maxpool2(x)

        ## block 3
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)

        ## block 4
        x = self.inception4a(x)
        if self.use_aux:
            aux1 = self.aux1(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        if self.use_aux:
            aux2 = self.aux2(x)
        x = self.inception4e(x)
        x = self.maxpool4(x)

        ## block 5
        x = self.inception5a(x)
        x = self.inception5b(x)

        ## block 6
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)

        if self.use_aux:
            return x, aux1, aux2
        else:
            return x